# 02 · Event representation

**pragmatiq series, notebook 2 of 4 after the [tour](00_blueprint_v2_tour.ipynb). Slot 1: how one event becomes
tokens, and how time is represented.**

A transaction is a small record: an amount, a merchant, a category code, a
channel, and a timestamp. A transformer needs integer tokens. This notebook
opens the tokenizer that ships with the pretrained checkpoint. It shows what
one event becomes, the four ways a field value turns into a token, how
identifiers were made cheap to model, and how time enters without being a
token at all.

The kit keeps the *token scheme* (how fields become tokens) separate from the
*sequence structure* (how tokens are grouped), so any scheme can feed any
structure that can find each event's boundary.

About 3 minutes on the lab GPU, on the tour's 4,000-user sample.

> **Smoke mode.** CI runs this notebook with `TFM_NOTEBOOK_SMOKE=1`: a
> generated 120-user book, an untrained model, and CPU only. Every cell runs,
> but the numbers and plots are meaningless. The first code cell prints which
> mode you are in.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfm.runtime.lab import fetch_lab_assets, prepare_lab_data
from tfm.training.reproducibility import seed_everything

GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = [GREEN, "#0074DF", "#F2A900", "#5D1682", "#E52020", DARK, GREY, "#00A3A3"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

assets = fetch_lab_assets("02")
config = assets.config
seed_everything(config.seed)
print(assets.describe())

In [ ]:
data = prepare_lab_data(assets)
tokenizer = data.tokenizer
names = tokenizer.id_to_token
print(f"{tokenizer.name}: layout {tokenizer.layout!r}, {tokenizer.vocab_size:,} tokens")

## One event, as tokens

The key-value tokenizer writes every field as a pair: a *key* token that
names the field (`KEY_amount`) and one or more *value* tokens
(`VAL_amount_b9`). An event is the list of its pairs, so events from
different sources simply have different keys. The same record also carries
each event's calendar fields and timestamps, which are not tokens (see
below).

In [ ]:
sequence = max(data.test_sequences, key=lambda s: len({e.categorical.get("source") for e in s.events}))
record = tokenizer.encode(sequence)
n_events = len(record.event_offsets) - 1
encoded = sequence.events[-n_events:]
sources = [e.categorical.get("source") for e in encoded]
for j in [sources.index(s) for s in dict.fromkeys(sources)][:3]:
    start, end = record.event_offsets[j], record.event_offsets[j + 1]
    print(f"event {j}: {sources[j]} at {encoded[j].event_timestamp:%Y-%m-%d %H:%M}")
    for key, value in zip(record.key_ids[start:end], record.value_ids[start:end]):
        field = names[key].removeprefix("KEY_")
        print(f"    {names[key]:<18} {names[value]:<30} {tokenizer.field_kind.get(field, '')}")

## Four ways a value becomes a token

Each key is assigned a *kind* when the tokenizer is fitted on the train side:

| Kind | Used for | Tokens |
|---|---|---|
| `categorical` | fields with few distinct values (`txn_type`, `country`, `mcc`) | one per value seen in training, plus `[UNK]` |
| `numeric` | numbers (`amount`) | 16 percentile buckets `VAL_amount_b1`…`b16`, and `b0` for zero |
| `hashed` | identifiers with huge cardinality (`merchant`, `device_id`, `counterparty`) | 4,096 buckets per key |
| bytes | any other free text | one token per UTF-8 byte |

The vocabulary below is dominated by the hash buckets. Each hashed key costs
a fixed 4,096 tokens however many merchants the book has, and a merchant
never seen in training still gets a real token rather than `[UNK]`.

In [ ]:
keys = sorted(tokenizer.key_vocab, key=len, reverse=True)
per_key = Counter()
for token in tokenizer.token_to_id:
    if token.startswith("VAL_"):
        per_key[next((k for k in keys if token.startswith(f"VAL_{k}_")), "other")] += 1
    elif token.startswith("BYTE_"):
        per_key["(bytes)"] += 1
kind_colour = {"hashed": GREEN, "categorical": "#0074DF", "numeric": "#F2A900"}
shown = per_key.most_common()
fig, ax = plt.subplots(figsize=(11, 0.28 * len(shown) + 1.2))
ax.barh([k for k, _ in shown][::-1], [n for _, n in shown][::-1],
        color=[kind_colour.get(tokenizer.field_kind.get(k), GREY) for k, _ in shown][::-1])
ax.set_xscale("log")
ax.set_xlabel("value tokens (log scale)")
ax.set_title("Vocabulary by key (green: hashed, blue: categorical, amber: numeric, grey: other)")
plt.tight_layout()
plt.show()

### Numbers become percentile buckets

Bucket edges are the train side's percentiles, so each of the 16 buckets
holds about one sixteenth of the training amounts. A £3 coffee and a £3,000
rent payment are both one token, and the model gets resolution where the data
is dense.

In [ ]:
edges = tokenizer.binners["amount"].edges
amounts = np.array([e.numeric["amount"] for s in data.train_sequences for e in s.events
                    if e.numeric.get("amount", 0.0) > 0])
amount_key = tokenizer.key_vocab["amount"]
bucket_counts = Counter()
for seq in data.test_sequences[:300]:
    rec = tokenizer.encode(seq)
    bucket_counts.update(names[v] for k, v in zip(rec.key_ids, rec.value_ids) if k == amount_key)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.6))
left.hist(amounts, bins=np.geomspace(max(amounts.min(), 0.01), amounts.max(), 60), color=GREY)
for edge in edges[edges > 0]:
    left.axvline(edge, color=GREEN, lw=0.8)
left.set_xscale("log")
left.set_title("Positive train-side amounts and the bucket edges (green)")
left.set_xlabel("amount (log scale)")
ordered = sorted(bucket_counts, key=lambda t: int(t.rsplit("_b", 1)[-1]))
right.bar(range(len(ordered)), [bucket_counts[t] for t in ordered], color=GREEN)
right.set_xticks(range(len(ordered)), [t.rsplit("_", 1)[-1] for t in ordered], rotation=90)
right.set_title("How often each amount bucket occurs (test side)")
plt.tight_layout()
plt.show()

The buckets are fitted on the train period and counted here on the test
period, so the right panel is flat only as far as spending did not drift
between the two.

### Identifiers are hashed

An identifier's token is `hash(key, value) mod 4096`, computed with a fixed
BLAKE2 digest so every process agrees. Buckets are shared (*collisions*).
The book has 50,000 merchants, so a merchant token means "one of about a
dozen merchants". The event's category, amount, and channel narrow it
further. Devices and counterparties are fewer, and most keep a bucket to
themselves.

In [ ]:
from tfm.features.tokenizers.key_value_time import hash_bucket

buckets = tokenizer.config.hash_buckets
collisions = {}
for key in ("merchant", "device_id", "counterparty"):
    values = {e.categorical[key] for e in data.events if key in e.categorical}
    occupancy = Counter(hash_bucket(key, v, buckets) for v in values)
    shared = sum(n for n in occupancy.values() if n > 1)
    collisions[key] = {"distinct values in sample": len(values), "buckets used": len(occupancy),
                       "values sharing a bucket": shared,
                       "share sharing": shared / max(len(values), 1)}
pd.DataFrame(collisions).T.round(3)

A shared bucket is still far more learnable than a spelled-out name. The
alternative, a token per merchant, costs a 50,000-row embedding table and a
50,000-way softmax in the next-event head. The search tried 8,192 buckets per
key: the full-vocabulary softmax did not fit in memory beside the prepared
book.

### Why hashing mattered

An earlier version of this model spelled these identifiers byte by byte, up
to 32 tokens each. Measured on 2,000 users of the same book, hashing took an
event from 20 tokens at the median to 7 (and from 30 to 9 at the 99th
percentile), so the same GPU reads about three times as many events per
second. The vocabulary grew from 535 tokens to 12,870, almost all of it hash
buckets. Spelled in bytes, the merchant had not been learned at all: its
next-event loss sat at the unigram floor for every byte. As one token, it is
learned slowly (notebook 03b).

### Tokens per event in this sample

With identifiers hashed, no event comes near the 40-token cap
(`data.max_tokens`), so nothing is truncated.

In [ ]:
lengths = {}
for seq in data.test_sequences[:400]:
    rec = tokenizer.encode(seq)
    per_event = np.diff(rec.event_offsets)
    for event, n in zip(seq.events[-len(per_event):], per_event):
        lengths.setdefault(event.categorical.get("source", "unknown"), []).append(int(n))
fig, ax = plt.subplots(figsize=(11, 3.4))
top = sorted(lengths, key=lambda s: -len(lengths[s]))[:6]
ax.hist([lengths[s] for s in top], bins=np.arange(0.5, 16.5), label=top, stacked=True)
ax.axvline(config.data.max_tokens, color=DARK, ls="--")
ax.set_xlabel("tokens per event")
ax.set_ylabel("events")
ax.set_title("Tokens per event by source (test side, 400 users)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Time is not a token

Each event also carries its calendar (hour, day of week, day of month),
embedded and added to the event's vector. The decoder then rotates attention
by elapsed time with **TimeRoPE**, a rotary position encoding. An event's
position is not its index but `8·ln(1 + Δt/8)`, where `Δt` is the seconds
from the event to the sequence's last event. Two events attend according to
the difference of their positions. The model can then tell a burst of
purchases from a monthly bill. The log keeps resolution for short gaps and
compresses long ones.

In [ ]:
from tfm.features.tokenizers.key_value_time import time_encode

gaps = np.array([(b.event_timestamp - a.event_timestamp).total_seconds()
                 for s in data.test_sequences[:400] for a, b in zip(s.events, s.events[1:])])
seconds = np.geomspace(1, 400 * 86400, 200)
marks = {"1 min": 60, "1 hour": 3600, "1 day": 86400, "1 week": 7 * 86400, "30 days": 30 * 86400}

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.6))
left.plot(seconds, [time_encode(s) for s in seconds], color=GREEN, lw=2)
for label, s in marks.items():
    left.axvline(s, color=GREY, ls=":", lw=1)
    left.annotate(label, (s, time_encode(s)), textcoords="offset points", xytext=(4, -12), fontsize=8)
left.set_xscale("log")
left.set_xlabel("Δt (seconds, log scale)")
left.set_ylabel("TimeRoPE input")
left.set_title("The log-time transform")
right.hist(gaps[gaps > 0], bins=np.geomspace(1, max(gaps.max(), 2), 60), color=GREEN)
for label, s in marks.items():
    right.axvline(s, color=GREY, ls=":", lw=1)
right.set_xscale("log")
right.set_xlabel("seconds between consecutive events (log scale)")
right.set_title("Gaps between consecutive events")
plt.tight_layout()
plt.show()

In [ ]:
for i in np.linspace(0, n_events - 1, 6).astype(int):
    print(f"event {i:>3}: hour {record.hour[i]:>2}, weekday {record.dow[i]}, day {record.dom[i]:>2}, "
          f"position {record.time_log[i]:6.2f}")

The gaps are bimodal. Seconds separate the taps of one app session; hours to
a day separate one session or purchase from the next. A model indexed by
event number would see both as one step apart.

## The one constraint: find the event boundary

The kit has two other token schemes. `event_fields` encodes v1's 12 fixed
transaction fields as key-value pairs. `flat_key_value` writes one flat
stream (`[USR]`, the profile, then `[EVT]` and the fields of each event) for
a flat decoder like v1's.

Blueprint v2 compresses each event on its own, so it must know where every
event starts. A flat stream has no such boundary, and the kit refuses the
pairing before training (rule 4 of the four composition
rules that notebook 03b lists).

In [ ]:
from tfm.api.validation import validate_tokenizer_layout
from tfm.config import RecipeConfig
from tfm.contracts.errors import IncompatibleCompositionError
from tfm.runtime.data import tokenizer_for
from tfm.runtime.pipeline import build_model

flat = config.model_dump(mode="json")
flat["features"].update(tokenizer="flat_key_value", hashed_keys=[])
flat_tokenizer = tokenizer_for(RecipeConfig.model_validate(flat))
model = build_model(config, tokenizer.vocab_size)
try:
    validate_tokenizer_layout(flat_tokenizer, model.spec)
except IncompatibleCompositionError as error:
    print("Refused:", error)

### Exercises

1. Recount the hash collisions with 1,024 buckets instead of 4,096 (`hash_bucket(key, v, 1024)`). How many merchants share a bucket now, and what would that cost the next-event head?
2. Encode the same event with a few different amounts and find which bucket each lands in. Where are the buckets narrow, and why?
3. The flat `flat_key_value` scheme was refused for Blueprint v2. Which structure in the Blueprint v1 notebooks (02 to 05) would accept it, and why does that one need no event boundary?

## Where this lives in the kit

This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the
same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2), and no
reusable logic lives here.

The tokenizer is `tfm.features.tokenizers.key_value_time`; its fitted state is
`tokenizer.json` in the checkpoint directory. `features.hashed_keys` and
`features.hash_buckets` in the checkpoint's `config.yaml` choose which keys are hashed.

Next: [03b · Architecture and objective](03b_blueprint_v2_decoder.ipynb).